In [1]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoModel, AutoTokenizer

/home/henry/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## load data

In [2]:
df = pd.read_csv("../data/predictions/pred_tweets.csv")
df

,text,label,predicted_scores,predicted_label
0,$BYND - JPMorgan reels in expectations on Beyo...,negative,"{'negative': 0.7128058671951294, 'positive': 0...",negative
1,$CCL $RCL - Nomura points to bookings weakness...,negative,"{'negative': 0.9685064554214478, 'neutral': 0....",negative
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",negative,"{'negative': 0.9608473777770996, 'positive': 0...",negative
3,$ESS: BTIG Research cuts to Neutral https://t....,negative,"{'neutral': 0.5311363935470581, 'negative': 0....",neutral
4,$FNKO - Funko slides after Piper Jaffray PT cu...,negative,"{'negative': 0.935271680355072, 'neutral': 0.0...",negative
...,...,...,...,...
9538,The Week's Gainers and Losers on the Stoxx Eur...,neutral,"{'negative': 0.8407225608825684, 'neutral': 0....",negative
9539,Tupperware Brands among consumer gainers; Unil...,neutral,"{'neutral': 0.5108155608177185, 'positive': 0....",neutral
9540,vTv Therapeutics leads healthcare gainers; Myo...,neutral,"{'positive': 0.5435549020767212, 'neutral': 0....",positive
9541,"WORK, XPO, PYX and AMKR among after hour movers",neutral,"{'neutral': 0.9413087964057922, 'positive': 0....",neutral


In [3]:
N_CLUSTERS = 10


df["scores"] = df["predicted_scores"].apply(ast.literal_eval)
df["confidence"] = df["scores"].apply(max)
df["is_error"] = df["label"] != df["predicted_label"]
print(f"{len(df)} tweets. {df["is_error"].mean()} missclassified.")

9543 tweets. 0.28785497223095463 missclassified.


## embeddings

In [ ]:
tok = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModel.from_pretrained("ProsusAI/finbert").eval()

df["text"] = df["text"].fillna("").astype(str)

def embed(texts, bs=64):
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), bs):
            enc = tok(texts[i:i + bs], padding=True, truncation=True,
                      max_length=128, return_tensors="pt")
            h = model(**enc).last_hidden_state.float()
            m = enc["attention_mask"].unsqueeze(-1).float()
            out.append(((h * m).sum(1) / m.sum(1).clamp(min=1e-9)).numpy())
    return np.vstack(out)

emb = embed(df["text"].tolist())

# diagnose: which rows produced NaN?
bad = np.isnan(emb).any(axis=1)
print("rows with NaN embeddings:", bad.sum())
print(df.loc[bad, "text"].head())

# drop them so the rest of the pipeline works
df = df.loc[~bad].reset_index(drop=True)

df["cluster"] = KMeans(N_CLUSTERS, n_init=10, random_state=0).fit_predict(emb)
df[["x,", "y"]] = PCA(2, random_state=0).fit_transform(emb)

tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
X = tfidf.fit_transform(df["text"])
terms = tfidf.get_feature_names_out()

for c, g in df.groupby("cluster"):
    top = X[g.index].mean(axis=0).A1.argsort()[::-1][:8]
DATA = ""
df[df["is_error"]].sort_values("confidence", ascending=False).drop(columns="scores").to_csv(DATA / "missclassified.csv", index=False)

fig, ax = plt.subplots(1,2, figsize=(15,6))
ax[0].scatter(df.x, df.y, c=df.cluster,cmap="tab10",s=8)
ax[0].set_title("Clusters")
ax[1].scatter(df.x, df.y, c="lightgrey", s=8)
ax[1].scatter(df[df.is_error].x, df[df.is_error].y, c="red", s=12)
ax[1].set_title("Misclassified (red)")
plt.savefig(DATA / "clusters.png", dpi=150)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5540.29it/s]
BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.weight            | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ValueError: Input X contains NaN.
KMeans does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values